In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from scipy.stats import shapiro, spearmanr, kruskal, f_oneway, chi2_contingency, pointbiserialr, mannwhitneyu, ttest_ind
import warnings
warnings.filterwarnings('ignore')
import os

# CONFIGURACIÓN DE RUTAS MEJORADA
data_path = r"C:\Users\PC\Desktop\ProjecteData\Equip_15\Data\RRHH_220925_clean.parquet"
output_path = r"C:\Users\PC\Desktop\ProjecteData\Equip_15\Data\Resultados_Analisis_Completo_220925"

# Crear directorio de salida
os.makedirs(output_path, exist_ok=True)

print("="*80)
print("ANÁLISIS HÍBRIDO COMPLETO - AUSENTISMO LABORAL")
print("="*80)
print(f"Ruta de salida: {output_path}")

# 1. CARGA Y PREPARACIÓN DE DATOS
print("\n1. CARGANDO Y PREPARANDO DATOS...")
print("-" * 50)

df = pd.read_parquet(data_path)
print(f"✓ Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")

# Eliminar ID y crear variable binaria de ausentismo alto
df_clean = df.drop(columns=['ID'], errors='ignore')

# Crear variable binaria para análisis de grupos
median_absenteeism = df_clean['Absenteeism_hours'].median()
df_clean['Ausentismo_Alto'] = (df_clean['Absenteeism_hours'] > median_absenteeism).astype(int)

# Variables para análisis
numeric_vars = ['Transportation_expense', 'Distance_Residence_Work', 'Service_time', 
                'Age', 'Work_load_Average_day', 'Hit_target', 'Disciplinary_failure', 
                'Son', 'Social_drinker', 'Social_smoker', 'Pet', 'Weight', 'Height', 
                'Body_mass_index', 'Education_numeric', 'Month_absence_order', 
                'Day_week_order', 'Seasons_order']

categorical_vars = ['Reason_absence', 'Month_absence', 'Day_week', 'Seasons', 'Education']
target_continuous = 'Absenteeism_hours'
target_binary = 'Ausentismo_Alto'

# 2. ANÁLISIS DE NORMALIDAD (Shapiro-Wilk)
print("\n2. ANÁLISIS DE NORMALIDAD (Shapiro-Wilk)...")
print("-" * 50)

normality_results = []
for var in numeric_vars + [target_continuous]:
    if var in df_clean.columns:
        data = df_clean[var].dropna()
        if len(data) > 3:
            stat, p_value = shapiro(data)
            normality_results.append({
                'Variable': var,
                'Estadistico_Shapiro': round(stat, 6),
                'p_value_Shapiro': round(p_value, 6),
                'Es_Normal': p_value > 0.05,
                'n': len(data)
            })

df_normality = pd.DataFrame(normality_results)
print(df_normality.to_string(index=False))

# 3. FUNCIONES AUXILIARES MEJORADAS
def clasificar_correlacion(corr):
    """Clasificar la fuerza de la correlación"""
    abs_corr = abs(corr)
    if abs_corr < 0.1:
        return "Muy débil", 1
    elif abs_corr < 0.3:
        return "Débil", 2
    elif abs_corr < 0.5:
        return "Moderada", 3
    elif abs_corr < 0.7:
        return "Fuerte", 4
    else:
        return "Muy fuerte", 5

def interpretar_efecto_ausentismo(corr, variable):
    """Interpretar el efecto en el ausentismo"""
    if abs(corr) < 0.1:
        return "Efecto mínimo"
    elif corr > 0:
        return f"Aumenta ausentismo"
    else:
        return f"Disminuye ausentismo"

def clasificar_efecto_cohen(d):
    """Clasificar el tamaño del efecto de Cohen"""
    abs_d = abs(d)
    if abs_d < 0.2:
        return "Muy pequeño"
    elif abs_d < 0.5:
        return "Pequeño"
    elif abs_d < 0.8:
        return "Mediano"
    else:
        return "Grande"

def cohens_d(x, y):
    """Calcular Cohen's d para tamaño del efecto"""
    nx = len(x)
    ny = len(y)
    dof = nx + ny - 2
    pooled_std = np.sqrt(((nx-1)*np.std(x, ddof=1)**2 + (ny-1)*np.std(y, ddof=1)**2) / dof)
    return (np.mean(x) - np.mean(y)) / pooled_std

# 4. ANÁLISIS COMPLETO PARA VARIABLES NUMÉRICAS
print("\n4. ANÁLISIS COMPLETO PARA VARIABLES NUMÉRICAS...")
print("-" * 50)

results_numeric = []

for var in numeric_vars:
    if var in df_clean.columns:
        # Preparar datos para grupos
        data_high = df_clean[df_clean[target_binary] == 1][var].dropna()
        data_low = df_clean[df_clean[target_binary] == 0][var].dropna()
        
        if len(data_high) > 5 and len(data_low) > 5:
            # CORRELACIÓN SPEARMAN (continuo-continuo)
            spearman_corr, spearman_p = spearmanr(df_clean[var].dropna(), 
                                                df_clean[target_continuous].dropna())
            fuerza_spearman, fuerza_num = clasificar_correlacion(spearman_corr)
            efecto_ausentismo = interpretar_efecto_ausentismo(spearman_corr, var)
            direccion_spearman = "Positiva" if spearman_corr > 0 else "Negativa"
            
            # POINT-BISERIAL (continuo-binario)
            pointbiserial_corr, pointbiserial_p = pointbiserialr(df_clean[var].dropna(), 
                                                               df_clean[target_binary].dropna())
            fuerza_pointbiserial, _ = clasificar_correlacion(pointbiserial_corr)
            
            # MANN-WHITNEY U (no paramétrico)
            mw_stat, mw_p = mannwhitneyu(data_high, data_low, alternative='two-sided')
            
            # T-TEST (paramétrico)
            t_stat, t_p = ttest_ind(data_high, data_low, equal_var=False)
            
            # COHEN'S D (tamaño del efecto)
            d_effect = cohens_d(data_high, data_low)
            tamaño_efecto = clasificar_efecto_cohen(d_effect)
            
            # Determinar qué test usar para diferencias de medias
            es_normal_var = False
            if var in df_normality['Variable'].values:
                es_normal_var = df_normality[df_normality['Variable'] == var]['Es_Normal'].values[0]
            
            test_medias = "T-test" if es_normal_var else "Mann-Whitney"
            p_value_medias = t_p if es_normal_var else mw_p
            sig_medias = p_value_medias < 0.05
            
            results_numeric.append({
                'Variable': var,
                'Tipo': 'Numérica',
                # Correlaciones
                'Correlacion_Spearman': round(spearman_corr, 6),
                'p_value_Spearman': round(spearman_p, 6),
                'Fuerza_Spearman': fuerza_spearman,
                'Fuerza_Num': fuerza_num,
                'Direccion_Spearman': direccion_spearman,
                'Efecto_Ausentismo': efecto_ausentismo,
                'Significativa_Spearman': spearman_p < 0.05,
                
                'Correlacion_PointBiserial': round(pointbiserial_corr, 6),
                'p_value_PointBiserial': round(pointbiserial_p, 6),
                'Fuerza_PointBiserial': fuerza_pointbiserial,
                'Significativa_PointBiserial': pointbiserial_p < 0.05,
                
                # Tests de diferencias
                'Test_Medias': test_medias,
                'p_value_Medias': round(p_value_medias, 6),
                'Significativa_Medias': sig_medias,
                
                # Tamaño del efecto
                'Cohens_d': round(d_effect, 6),
                'Tamaño_Efecto': tamaño_efecto,
                
                # Estadísticas descriptivas por grupo
                'Media_Alto': round(data_high.mean(), 3),
                'Media_Bajo': round(data_low.mean(), 3),
                'Diferencia_Medias': round(data_high.mean() - data_low.mean(), 3),
                'n_Alto': len(data_high),
                'n_Bajo': len(data_low),
                
                # Para ordenamiento
                'abs_corr': abs(spearman_corr),
                'abs_effect': abs(d_effect)
            })

df_results_numeric = pd.DataFrame(results_numeric)

# AÑADIR COLUMNAS ABSOLUTAS PARA ORDENAMIENTO
if not df_results_numeric.empty:
    print(df_results_numeric[['Variable', 'Correlacion_Spearman', 'Fuerza_Spearman', 
                             'Significativa_Spearman', 'Cohens_d', 'Tamaño_Efecto']].to_string(index=False))
else:
    print("No hay variables numéricas para analizar")

# 5. ANÁLISIS COMPLETO PARA VARIABLES CATEGÓRICAS
print("\n5. ANÁLISIS COMPLETO PARA VARIABLES CATEGÓRICAS...")
print("-" * 50)

results_categorical = []

for var in categorical_vars:
    if var in df_clean.columns:
        try:
            # KRUSKAL-WALLIS (no paramétrico)
            groups_kw = [group[target_continuous].values for name, group in df_clean.groupby(var) 
                        if len(group) > 5]
            
            if len(groups_kw) > 1:
                kw_stat, kw_p = kruskal(*groups_kw)
                sig_kw = kw_p < 0.05
                
                # ANOVA (paramétrico)
                groups_anova = [df_clean[df_clean[var] == cat][target_continuous].dropna() 
                              for cat in df_clean[var].unique() if len(df_clean[df_clean[var] == cat]) > 5]
                
                if len(groups_anova) > 1:
                    anova_stat, anova_p = f_oneway(*groups_anova)
                    sig_anova = anova_p < 0.05
                else:
                    anova_stat, anova_p, sig_anova = np.nan, np.nan, False
                
                # CHI-CUADRADO para asociación con variable binaria
                contingency_table = pd.crosstab(df_clean[var], df_clean[target_binary])
                if contingency_table.shape[0] > 1 and contingency_table.shape[1] > 1:
                    chi2_stat, chi2_p, dof, expected = chi2_contingency(contingency_table)
                    sig_chi2 = chi2_p < 0.05
                else:
                    chi2_stat, chi2_p, sig_chi2 = np.nan, np.nan, False
                
                # Determinar test principal basado en normalidad
                es_normal_target = False
                if target_continuous in df_normality['Variable'].values:
                    es_normal_target = df_normality[df_normality['Variable'] == target_continuous]['Es_Normal'].values[0]
                
                test_principal = 'ANOVA' if es_normal_target else 'Kruskal-Wallis'
                estadistico_principal = anova_stat if es_normal_target else kw_stat
                p_value_principal = anova_p if es_normal_target else kw_p
                sig_principal = sig_anova if es_normal_target else sig_kw
                
                results_categorical.append({
                    'Variable': var,
                    'Tipo': 'Categórica',
                    'Test_Principal': test_principal,
                    'Estadistico_Principal': round(estadistico_principal, 6) if not np.isnan(estadistico_principal) else np.nan,
                    'p_value_Principal': round(p_value_principal, 6) if not np.isnan(p_value_principal) else np.nan,
                    'Significativa_Principal': sig_principal,
                    'Chi2_Estadistico': round(chi2_stat, 6) if not np.isnan(chi2_stat) else np.nan,
                    'p_value_Chi2': round(chi2_p, 6) if not np.isnan(chi2_p) else np.nan,
                    'Significativa_Chi2': sig_chi2,
                    'n_Categorias': len(df_clean[var].unique()),
                    'n_Total': len(df_clean[var].dropna())
                })
        except Exception as e:
            print(f"✗ Error en {var}: {e}")

df_results_categorical = pd.DataFrame(results_categorical)
if not df_results_categorical.empty:
    print(df_results_categorical[['Variable', 'Test_Principal', 'p_value_Principal', 
                                'Significativa_Principal', 'Significativa_Chi2']].to_string(index=False))
else:
    print("No se pudieron realizar tests para variables categóricas")

# 6. VISUALIZACIONES COMPLETAS (TODOS LOS GRÁFICOS)
print("\n6. GENERANDO TODAS LAS VISUALIZACIONES...")
print("-" * 50)

# A. HEATMAP DE CORRELACIONES SPEARMAN (ORIGINAL)
plt.figure(figsize=(16, 14))
corr_vars = [v for v in numeric_vars if v in df_clean.columns] + [target_continuous]
if len(corr_vars) > 1:
    corr_matrix = df_clean[corr_vars].corr(method='spearman')
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

    sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0, 
                square=True, fmt='.3f', cbar_kws={'shrink': 0.8})
    plt.title('MATRIZ DE CORRELACIÓN SPEARMAN - AUSENTISMO LABORAL\n(Todas las variables numéricas)', 
              fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(f'{output_path}/1_heatmap_correlaciones.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Heatmap de correlaciones guardado")

# B. GRÁFICO DE IMPORTANCIA DE VARIABLES (COMBINADO) (ORIGINAL)
if not df_results_numeric.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

    # Importancia por correlación Spearman
    corr_data = df_results_numeric.sort_values('abs_corr', ascending=True)
    
    colors = ['red' if sig else 'gray' for sig in corr_data['Significativa_Spearman']]
    ax1.barh(corr_data['Variable'], corr_data['abs_corr'], color=colors, alpha=0.7)
    ax1.set_xlabel('Correlación Absoluta (Spearman)')
    ax1.set_title('IMPORTANCIA POR CORRELACIÓN\n(Rojo = Significativo, p < 0.05)', fontweight='bold')
    ax1.grid(axis='x', alpha=0.3)

    # Importancia por tamaño del efecto (Cohen's d)
    effect_data = df_results_numeric.sort_values('abs_effect', ascending=True)
    
    colors_effect = ['red' if sig else 'gray' for sig in effect_data['Significativa_Medias']]
    ax2.barh(effect_data['Variable'], effect_data['abs_effect'], color=colors_effect, alpha=0.7)
    ax2.set_xlabel('Tamaño del Efecto Absoluto (Cohen\'s d)')
    ax2.set_title('IMPORTANCIA POR TAMAÑO DEL EFECTO\n(Rojo = Diferencias Significativas)', fontweight='bold')
    ax2.grid(axis='x', alpha=0.3)

    plt.suptitle('COMPARACIÓN DE IMPORTANCIA DE VARIABLES', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{output_path}/2_importancia_variables.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Gráfico de importancia de variables guardado")

# C. SCATTER PLOTS PARA VARIABLES MÁS SIGNIFICATIVAS (ORIGINAL)
if not df_results_numeric.empty:
    sig_vars_df = df_results_numeric[df_results_numeric['Significativa_Spearman']]
    if not sig_vars_df.empty:
        sig_vars = sig_vars_df.nlargest(6, 'abs_corr')['Variable'].tolist()
        
        n_vars = len(sig_vars)
        n_cols = min(3, n_vars)
        n_rows = (n_vars + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
        if n_rows == 1 and n_cols == 1:
            axes = [axes]
        elif n_rows == 1:
            axes = axes
        else:
            axes = axes.ravel()
        
        for i, var in enumerate(sig_vars):
            if i < len(axes):
                sns.regplot(data=df_clean, x=var, y=target_continuous, ax=axes[i], 
                           scatter_kws={'alpha':0.6, 's':30}, line_kws={'color':'red'}, ci=95)
                
                stats_row = df_results_numeric[df_results_numeric['Variable'] == var].iloc[0]
                corr = stats_row['Correlacion_Spearman']
                p_val = stats_row['p_value_Spearman']
                fuerza = stats_row['Fuerza_Spearman']
                direccion = stats_row['Direccion_Spearman']
                
                axes[i].set_title(f'{var}\nρ = {corr:.3f} ({fuerza}, {direccion})', 
                                fontweight='bold', fontsize=10)
                axes[i].set_xlabel(var)
                axes[i].set_ylabel('Horas de Ausentismo')
        
        # Ocultar ejes vacíos
        for j in range(len(sig_vars), n_rows * n_cols):
            if j < len(axes):
                axes[j].set_visible(False)
        
        plt.suptitle('RELACIÓN DE VARIABLES MÁS SIGNIFICATIVAS CON AUSENTISMO', 
                     fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig(f'{output_path}/3_scatter_variables_significativas.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("✓ Scatter plots de variables significativas guardados")

# D. BOXPLOTS PARA VARIABLES CATEGÓRICAS SIGNIFICATIVAS (ORIGINAL)
if not df_results_categorical.empty:
    sig_cat_vars = df_results_categorical[df_results_categorical['Significativa_Principal']]['Variable'].tolist()
    
    if sig_cat_vars:
        n_cat_vars = len(sig_cat_vars)
        n_cols = min(2, n_cat_vars)
        n_rows = (n_cat_vars + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 5*n_rows))
        if n_rows == 1 and n_cols == 1:
            axes = [axes]
        elif n_rows == 1:
            axes = axes
        else:
            axes = axes.ravel()
        
        for i, cat_var in enumerate(sig_cat_vars):
            if i < len(axes):
                order = df_clean.groupby(cat_var)[target_continuous].median().sort_values(ascending=False).index
                
                sns.boxplot(data=df_clean, x=cat_var, y=target_continuous, ax=axes[i], order=order)
                axes[i].set_title(f'Ausentismo por {cat_var}', fontweight='bold')
                axes[i].tick_params(axis='x', rotation=45)
                axes[i].set_xlabel(cat_var)
                axes[i].set_ylabel('Horas de Ausentismo')
        
        for j in range(len(sig_cat_vars), n_rows * n_cols):
            if j < len(axes):
                axes[j].set_visible(False)
        
        plt.suptitle('DISTRIBUCIÓN DE AUSENTISMO POR VARIABLES CATEGÓRICAS SIGNIFICATIVAS', 
                     fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig(f'{output_path}/4_boxplots_categoricas.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("✓ Boxplots de variables categóricas guardados")

# E. GRÁFICO DE CORRELACIONES MEJORADO - FUERZA Y DIRECCIÓN (NUEVO)
print("\nGENERANDO GRÁFICOS DE CORRELACIÓN MEJORADOS...")
if not df_results_numeric.empty:
    # Gráfico de barras de correlaciones
    df_plot = df_results_numeric.sort_values('abs_corr', ascending=True)
    
    plt.figure(figsize=(14, 10))
    
    # Definir colores basados en dirección y fuerza
    colors = []
    for _, row in df_plot.iterrows():
        if row['Correlacion_Spearman'] > 0:
            intensity = min(0.3 + row['Fuerza_Num'] * 0.15, 0.9)
            colors.append((intensity, 0.2, 0.2, 0.8))
        else:
            intensity = min(0.3 + row['Fuerza_Num'] * 0.15, 0.9)
            colors.append((0.2, 0.2, intensity, 0.8))
    
    # Crear gráfico de barras
    bars = plt.barh(df_plot['Variable'], df_plot['Correlacion_Spearman'], color=colors, alpha=0.8)
    
    # Añadir etiquetas de valores
    for i, (value, variable) in enumerate(zip(df_plot['Correlacion_Spearman'], df_plot['Variable'])):
        if value >= 0:
            plt.text(value + 0.01, i, f'{value:.3f}', va='center', ha='left', fontweight='bold')
        else:
            plt.text(value - 0.01, i, f'{value:.3f}', va='center', ha='right', fontweight='bold')
    
    # Línea vertical en 0
    plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
    
    # Personalizar
    plt.xlabel('Coeficiente de Correlación (Spearman)', fontsize=12, fontweight='bold')
    plt.title('IMPACTO DE VARIABLES EN EL AUSENTISMO LABORAL\n(Fuerza y Dirección de Correlación)', 
              fontsize=16, fontweight='bold', pad=20)
    
    # Leyenda personalizada
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor=(0.8, 0.2, 0.2, 0.8), label='AUMENTA AUSENTISMO (Correlación +)'),
        Patch(facecolor=(0.2, 0.2, 0.8, 0.8), label='DISMINUYE AUSENTISMO (Correlación -)'),
        Patch(facecolor=(0.9, 0.9, 0.9, 0.8), label='○ Muy débil (|ρ| < 0.1)'),
        Patch(facecolor=(0.7, 0.7, 0.7, 0.8), label='○ Débil (0.1 ≤ |ρ| < 0.3)'),
        Patch(facecolor=(0.5, 0.5, 0.5, 0.8), label='○ Moderada (0.3 ≤ |ρ| < 0.5)'),
        Patch(facecolor=(0.3, 0.3, 0.3, 0.8), label='○ Fuerte (0.5 ≤ |ρ| < 0.7)'),
        Patch(facecolor=(0.1, 0.1, 0.1, 0.8), label='○ Muy fuerte (|ρ| ≥ 0.7)')
    ]
    
    plt.legend(handles=legend_elements, loc='lower right', bbox_to_anchor=(1, 0), 
               frameon=True, fancybox=True, shadow=True)
    
    # Añadir anotaciones de efecto
    for i, (_, row) in enumerate(df_plot.iterrows()):
        effect_text = row['Efecto_Ausentismo']
        color = 'darkred' if row['Correlacion_Spearman'] > 0 else 'darkblue'
        plt.annotate(effect_text, 
                    xy=(row['Correlacion_Spearman'], i),
                    xytext=(5, 0), textcoords='offset points',
                    ha='left', va='center', fontsize=9, color=color,
                    fontweight='bold', bbox=dict(boxstyle='round,pad=0.3', 
                    facecolor='lightyellow', alpha=0.7))
    
    plt.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    
    # Guardar gráfico
    plt.savefig(f'{output_path}/5_grafico_correlaciones_impacto.png', 
                dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Gráfico de correlaciones de impacto guardado")

# F. MAPA DE CALOR DE IMPACTO (NUEVO)
if not df_results_numeric.empty:
    # Crear matriz para heatmap
    heatmap_data = df_results_numeric[['Variable', 'Fuerza_Num', 'Correlacion_Spearman']].copy()
    heatmap_data['Impacto'] = heatmap_data['Correlacion_Spearman'].apply(
        lambda x: 'Positivo' if x > 0 else 'Negativo'
    )
    heatmap_data = heatmap_data.sort_values('Fuerza_Num', ascending=False)
    
    # Crear heatmap categórico
    plt.figure(figsize=(12, 8))
    
    # Mapear fuerzas a colores
    color_map = {
        'Muy débil': 'lightgray',
        'Débil': 'lightblue', 
        'Moderada': 'orange',
        'Fuerte': 'red',
        'Muy fuerte': 'darkred'
    }
    
    # Crear matriz de colores
    colors_heatmap = []
    for _, row in heatmap_data.iterrows():
        fuerza = df_results_numeric[df_results_numeric['Variable'] == row['Variable']]['Fuerza_Spearman'].values[0]
        if row['Impacto'] == 'Positivo':
            colors_heatmap.append(color_map.get(fuerza, 'gray'))
        else:
            # Para impacto negativo, usar tonos azules
            if fuerza == 'Muy débil': colors_heatmap.append('lightgray')
            elif fuerza == 'Débil': colors_heatmap.append('lightsteelblue')
            elif fuerza == 'Moderada': colors_heatmap.append('royalblue')
            elif fuerza == 'Fuerte': colors_heatmap.append('mediumblue')
            else: colors_heatmap.append('darkblue')
    
    # Crear scatter plot como heatmap
    scatter = plt.scatter(x=range(len(heatmap_data)), 
                         y=[1]*len(heatmap_data),
                         c=colors_heatmap, 
                         s=1000,  # Tamaño de los puntos
                         alpha=0.7)
    
    # Etiquetas
    plt.yticks([1], [''])
    plt.xticks(range(len(heatmap_data)), heatmap_data['Variable'], rotation=45, ha='right')
    
    # Añadir valores de correlación
    for i, (_, row) in enumerate(heatmap_data.iterrows()):
        plt.text(i, 1, f'{row["Correlacion_Spearman"]:.3f}', 
                ha='center', va='center', fontweight='bold', fontsize=10,
                color='white' if abs(row['Correlacion_Spearman']) > 0.3 else 'black')
    
    plt.title('MAPA DE IMPACTO - CORRELACIÓN CON AUSENTISMO\n(Tamaño del efecto y dirección)', 
              fontsize=14, fontweight='bold', pad=20)
    plt.xlabel('Variables')
    
    # Leyenda
    legend_elements = [
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='darkred', 
                  markersize=10, label='Muy fuerte (+)'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', 
                  markersize=10, label='Fuerte (+)'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='orange', 
                  markersize=10, label='Moderada (+)'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='lightblue', 
                  markersize=10, label='Débil (+)'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='darkblue', 
                  markersize=10, label='Muy fuerte (-)'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='mediumblue', 
                  markersize=10, label='Fuerte (-)'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='royalblue', 
                  markersize=10, label='Moderada (-)'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='lightsteelblue', 
                  markersize=10, label='Débil (-)')
    ]
    
    plt.legend(handles=legend_elements, bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    
    plt.savefig(f'{output_path}/6_mapa_calor_impacto.png', 
                dpi=300, bbox_inches='tight')
    plt.close()
    print("✓ Mapa de calor de impacto guardado")

# 7. EXPORTACIÓN DE TODOS LOS ARCHIVOS
print("\n7. EXPORTANDO TODOS LOS ARCHIVOS...")
print("-" * 50)

# Crear DataFrame final combinado
final_results = []

# Añadir variables numéricas
for _, row in df_results_numeric.iterrows():
    final_results.append({
        'Variable': str(row['Variable']),
        'Tipo': 'Numérica',
        'Correlacion_Spearman': float(row['Correlacion_Spearman']),
        'p_value_Spearman': float(row['p_value_Spearman']),
        'Fuerza_Correlacion': str(row['Fuerza_Spearman']),
        'Direccion_Correlacion': str(row['Direccion_Spearman']),
        'Efecto_Ausentismo': str(row['Efecto_Ausentismo']),
        'Significativa_Spearman': 'VERDADERO' if row['Significativa_Spearman'] else 'FALSO',
        'Correlacion_PointBiserial': float(row['Correlacion_PointBiserial']),
        'p_value_PointBiserial': float(row['p_value_PointBiserial']),
        'Significativa_PointBiserial': 'VERDADERO' if row['Significativa_PointBiserial'] else 'FALSO',
        'Test_Medias_Utilizado': str(row['Test_Medias']),
        'p_value_Medias': float(row['p_value_Medias']),
        'Significativa_Medias': 'VERDADERO' if row['Significativa_Medias'] else 'FALSO',
        'Cohens_d': float(row['Cohens_d']),
        'Tamaño_Efecto': str(row['Tamaño_Efecto']),
        'Media_Grupo_Alto': float(row['Media_Alto']),
        'Media_Grupo_Bajo': float(row['Media_Bajo']),
        'Diferencia_Medias': float(row['Diferencia_Medias'])
    })

# Añadir variables categóricas
for _, row in df_results_categorical.iterrows():
    final_results.append({
        'Variable': str(row['Variable']),
        'Tipo': 'Categórica',
        'Test_Principal': str(row['Test_Principal']) if pd.notna(row['Test_Principal']) else 'No aplica',
        'Estadistico_Principal': float(row['Estadistico_Principal']) if pd.notna(row['Estadistico_Principal']) else float('nan'),
        'p_value_Principal': float(row['p_value_Principal']) if pd.notna(row['p_value_Principal']) else float('nan'),
        'Significativa_Principal': 'VERDADERO' if row['Significativa_Principal'] else 'FALSO',
        'Chi2_Estadistico': float(row['Chi2_Estadistico']) if pd.notna(row['Chi2_Estadistico']) else float('nan'),
        'p_value_Chi2': float(row['p_value_Chi2']) if pd.notna(row['p_value_Chi2']) else float('nan'),
        'Significativa_Chi2': 'VERDADERO' if row['Significativa_Chi2'] else 'FALSO',
        'n_Categorias': int(row['n_Categorias']),
        # Para columnas numéricas, usar NaN en lugar de cadenas vacías
        'Correlacion_Spearman': float('nan'),
        'p_value_Spearman': float('nan'),
        'Fuerza_Correlacion': 'No aplica',
        'Direccion_Correlacion': 'No aplica',
        'Efecto_Ausentismo': 'No aplica',
        'Significativa_Spearman': 'No aplica',
        'Correlacion_PointBiserial': float('nan'),
        'p_value_PointBiserial': float('nan'),
        'Significativa_PointBiserial': 'No aplica',
        'Test_Medias_Utilizado': 'No aplica',
        'p_value_Medias': float('nan'),
        'Significativa_Medias': 'No aplica',
        'Cohens_d': float('nan'),
        'Tamaño_Efecto': 'No aplica',
        'Media_Grupo_Alto': float('nan'),
        'Media_Grupo_Bajo': float('nan'),
        'Diferencia_Medias': float('nan')
    })

df_final = pd.DataFrame(final_results)

# CONVERTIR COLUMNAS ESPECÍFICAS A TIPOS CORRECTOS
numeric_columns = ['Correlacion_Spearman', 'p_value_Spearman', 'Correlacion_PointBiserial', 
                  'p_value_PointBiserial', 'p_value_Medias', 'Cohens_d', 
                  'Media_Grupo_Alto', 'Media_Grupo_Bajo', 'Diferencia_Medias',
                  'Estadistico_Principal', 'p_value_Principal', 'Chi2_Estadistico', 'p_value_Chi2']

for col in numeric_columns:
    if col in df_final.columns:
        df_final[col] = pd.to_numeric(df_final[col], errors='coerce')

# EXPORTAR TODOS LOS ARCHIVOS
try:
    # Archivos principales
    df_final.to_csv(f'{output_path}/Resultados_Completos_Analisis_Hibrido.csv', 
                    index=False, encoding='utf-8-sig')
    df_final.to_parquet(f'{output_path}/Resultados_Completos_Analisis_Hibrido.parquet', 
                        index=False, engine='pyarrow')
    
    # Archivos adicionales
    df_normality.to_csv(f'{output_path}/Analisis_Normalidad.csv', index=False)
    df_clean.to_parquet(f'{output_path}/dataset_analisis_completo.parquet', index=False)
    
    # Archivos específicos de correlaciones
    df_results_numeric.to_csv(f'{output_path}/Correlaciones_Ausentismo_Detalladas.csv', 
                             index=False, encoding='utf-8-sig')
    
    # Resumen ejecutivo de correlaciones
    resumen_ejecutivo = df_results_numeric[['Variable', 'Correlacion_Spearman', 
                                          'Fuerza_Spearman', 'Efecto_Ausentismo',
                                          'Significativa_Spearman']]
    resumen_ejecutivo.to_csv(f'{output_path}/Resumen_Ejecutivo_Correlaciones.csv', 
                            index=False, encoding='utf-8-sig')
    
    print("✓ TODOS los archivos exportados correctamente:")
    print(f"  - Resultados_Completos_Analisis_Hibrido.csv")
    print(f"  - Resultados_Completos_Analisis_Hibrido.parquet")
    print(f"  - Analisis_Normalidad.csv")
    print(f"  - dataset_analisis_completo.parquet")
    print(f"  - Correlaciones_Ausentismo_Detalladas.csv")
    print(f"  - Resumen_Ejecutivo_Correlaciones.csv")
    
except Exception as e:
    print(f"✗ Error exportando archivos: {e}")

# 8. RESUMEN EJECUTIVO FINAL
print("\n" + "="*80)
print("RESUMEN EJECUTIVO COMPLETO")
print("="*80)

# Resumen de correlaciones
if not df_results_numeric.empty:
    print("\nVARIABLES QUE AUMENTAN EL AUSENTISMO (Correlación positiva):")
    print("-" * 60)
    positivas = df_results_numeric[df_results_numeric['Correlacion_Spearman'] > 0]
    for _, row in positivas.iterrows():
        sig = "✓" if row['Significativa_Spearman'] else "✗"
        print(f"{sig} {row['Variable']:30} ρ = {row['Correlacion_Spearman']:7.3f} ({row['Fuerza_Spearman']})")
    
    print("\nVARIABLES QUE DISMINUYEN EL AUSENTISMO (Correlación negativa):")
    print("-" * 60)
    negativas = df_results_numeric[df_results_numeric['Correlacion_Spearman'] < 0]
    for _, row in negativas.iterrows():
        sig = "✓" if row['Significativa_Spearman'] else "✗"
        print(f"{sig} {row['Variable']:30} ρ = {row['Correlacion_Spearman']:7.3f} ({row['Fuerza_Spearman']})")

print(f"\n✓ Gráficos generados:")
print(f"  1. 1_heatmap_correlaciones.png")
print(f"  2. 2_importancia_variables.png") 
print(f"  3. 3_scatter_variables_significativas.png")
print(f"  4. 4_boxplots_categoricas.png")
print(f"  5. 5_grafico_correlaciones_impacto.png")
print(f"  6. 6_mapa_calor_impacto.png")

print("\n" + "="*80)
print("ANÁLISIS COMPLETO FINALIZADO EXITOSAMENTE! 🎉")
print("="*80)
print(f"Todos los resultados guardados en: {output_path}")

ANÁLISIS HÍBRIDO COMPLETO - AUSENTISMO LABORAL
Ruta de salida: C:\Users\PC\Desktop\ProjecteData\Equip_15\Data\Resultados_Analisis_Completo_220925

1. CARGANDO Y PREPARANDO DATOS...
--------------------------------------------------
✓ Dataset cargado: 806 filas, 25 columnas

2. ANÁLISIS DE NORMALIDAD (Shapiro-Wilk)...
--------------------------------------------------
               Variable  Estadistico_Shapiro  p_value_Shapiro  Es_Normal   n
 Transportation_expense             0.947629              0.0      False 806
Distance_Residence_Work             0.880630              0.0      False 806
           Service_time             0.943835              0.0      False 806
                    Age             0.928054              0.0      False 806
  Work_load_Average_day             0.923185              0.0      False 806
             Hit_target             0.887959              0.0      False 806
   Disciplinary_failure             0.229883              0.0      False 806
              